In [0]:
from pyspark.sql import functions as F

# LECTURA DESDE LA CAPA PLATA
# Leemos nuestras dos tablas limpias de Unity Catalog
df_plata_ventas = spark.table("databricks_proyecto_jhon.lakehouse.plata_ventas")
df_plata_bcrp = spark.table("databricks_proyecto_jhon.lakehouse.plata_tipo_cambio_bcrp")

# MODELADO ESTRELLA (EL CRUCE)
# Hacemos un LEFT JOIN usando la columna "fecha" que ambas tablas comparten.
# Usamos LEFT para asegurarnos de no perder ninguna venta, incluso si el BCRP 
# no publicó el tipo de cambio ese día (como los domingos).
df_cruzado = df_plata_ventas.join(df_plata_bcrp, on="fecha", how="left")


# CREACIÓN DE MÉTRICAS DE NEGOCIO (Para Power BI)
df_fct_ventas = (df_cruzado
    # Calculamos el monto en dólares y lo redondeamos a 2 decimales
    .withColumn("monto_dolares", F.round(F.col("monto") / F.col("tipo_cambio"), 2))
    
    # Sello de auditoría final
    .withColumn("_fecha_modelo", F.current_timestamp())
    .drop("_fecha_limpieza") # Borramos la metadata de la capa anterior
    
    # Ordenamos las columnas para que se vea profesional en el dashboard
    .select(
        "id", "fecha", "id_empleado", "id_cliente", "estado", 
        "tipo_cambio", "monto", "monto_dolares", "_fecha_modelo"
    )
)

# CARGA A DELTA LAKE (Capa Oro)
(df_fct_ventas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("databricks_proyecto_jhon.lakehouse.fct_ventas")
)

print(f"✅ ¡Capa Oro finalizada! Modelo de ventas listo con {df_fct_ventas.count()} registros.")
display(df_fct_ventas)
print('🤖 ¡Prueba Webhook COMMIT Automatica de automatización exitosa! Jenkins inyectó esto.')